# Streamlined CRISPR Screen Analysis Tutorial

This notebook demonstrates how to run the on-disk analysis workflow provided by ``streamlined_crispr``. It walks through quality control, effect-size estimation via pseudo-bulk summaries, and differential expression testing without ever loading the full matrix into memory.


## Prerequisites

Install the project in editable mode with the optional ``test`` dependencies to obtain AnnData, NumPy, pandas, and SciPy::

    pip install -e .[test]

The commands below assume the repository root is the current working directory.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

from streamlined_crispr import (
    quality_control_summary,
    compute_average_log_expression,
    compute_pseudobulk_expression,
    wald_test,
    wilcoxon_test,
)


## Create a toy on-disk dataset

To keep the tutorial self-contained we generate a small synthetic AnnData object, assign perturbation labels, and write it to disk as an ``.h5ad`` file. In practice you would point the workflow at an existing file.


In [ ]:
data_dir = Path('tutorial_outputs')
data_dir.mkdir(exist_ok=True)
raw_path = data_dir / 'demo_raw.h5ad'

counts = np.array([
    [0, 1, 0, 2, 0],
    [3, 0, 0, 1, 0],
    [0, 2, 1, 0, 0],
    [1, 1, 0, 0, 4],
    [0, 0, 0, 3, 5],
    [2, 0, 0, 1, 6],
    [0, 0, 2, 0, 1],
    [1, 0, 0, 2, 0],
])
obs = pd.DataFrame({
    'perturbation': ['ctrl', 'ctrl', 'KO_TP53', 'KO_TP53', 'KO_BRCA1', 'KO_BRCA1', 'KO_TP53', 'KO_BRCA1']
})
var = pd.DataFrame({
    'gene_symbol': ['GATA3', 'MYC', 'KRAS', 'PTEN', 'BCL2']
})
var.index = var['gene_symbol']
adata = ad.AnnData(counts, obs=obs, var=var)
adata.write(raw_path)
raw_path


## Run quality control

The :func:`quality_control_summary` function filters low-quality cells, under-powered perturbations, and genes with limited expression. It returns boolean masks and writes a filtered ``.h5ad`` file that subsequent steps can reuse.


In [ ]:
qc_result = quality_control_summary(
    raw_path,
    min_genes=1,
    min_cells_per_perturbation=2,
    min_cells_per_gene=2,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbol',
    output_dir=data_dir,
    data_name='demo',
)
qc_result


In [ ]:
filtered_path = qc_result.filtered_path
filtered_path


We can also inspect how many cells and genes were retained.


In [ ]:
qc_summary = {
    'total_cells': qc_result.cell_mask.size,
    'kept_cells': int(qc_result.cell_mask.sum()),
    'total_genes': qc_result.gene_mask.size,
    'kept_genes': int(qc_result.gene_mask.sum()),
}
qc_summary


## Estimate perturbation effect sizes

After QC we can compute effect sizes using both supported strategies. The functions stream data from disk and write their own ``.h5ad`` artifacts for reproducibility.


In [ ]:
avg_effects = compute_average_log_expression(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_avg',
)
avg_effects


In [ ]:
pseudo_effects = compute_pseudobulk_expression(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    baseline_count=1.0,
    output_dir=data_dir,
    data_name='demo_pseudo',
)
pseudo_effects


## Differential expression testing

Finally, run the Wald and Wilcoxon tests. Each call returns a dictionary of :class:`DifferentialExpressionResult` objects keyed by perturbation and writes a compact ``.h5ad`` summary.


In [ ]:
wald_results = wald_test(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_wald',
)
{key: value.pvalue[:3] for key, value in wald_results.items()}


In [ ]:
wilcoxon_results = wilcoxon_test(
    filtered_path,
    perturbation_column='perturbation',
    control_label='ctrl',
    gene_name_column='gene_symbols',
    output_dir=data_dir,
    data_name='demo_wilcoxon',
)
{key: value.pvalue[:3] for key, value in wilcoxon_results.items()}


## Inspect generated files

Every major step leaves behind an ``.h5ad`` file in the output directory. This makes it easy to resume the analysis from any point or inspect intermediate results.


In [ ]:
sorted(path.name for path in data_dir.glob('*.h5ad'))
